<a href="https://colab.research.google.com/github/saranya3197/Predictive-Analytics/blob/recommending-products-to-users-based-on-other-users/Recommending_products_to_users_based_on_other_users.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
df = pd.read_csv("online_retail.csv", encoding='ISO-8859-1')
df = df[df['Quantity'] > 0]
df = df.dropna(subset=['CustomerID'])
df['CustomerID'] = df['CustomerID'].astype(int)
basket = df.pivot_table(index='CustomerID',
                        columns='StockCode',
                        values='Quantity',
                        aggfunc='sum',
                        fill_value=0)
similarity_matrix = cosine_similarity(basket)
user_similarity_df = pd.DataFrame(similarity_matrix, index=basket.index, columns=basket.index)
def recommend_products(customer_id, top_n_similar_users=3, top_k_products=5):
    if customer_id not in basket.index:
        return f"Customer ID {customer_id} not found in dataset."
    similarity_scores = user_similarity_df[customer_id].drop(customer_id)
    top_users = similarity_scores.sort_values(ascending=False).head(top_n_similar_users).index
    similar_users_data = basket.loc[top_users]
    mean_ratings = similar_users_data.mean()
    target_user_data = basket.loc[customer_id]
    products_to_recommend = mean_ratings[target_user_data == 0]
    recommended_products = products_to_recommend.sort_values(ascending=False).head(top_k_products)
    return pd.DataFrame({'Avg Quantity': recommended_products})
customer_id = 17850
print("Top product recommendations for Customer", customer_id)
print(recommend_products(customer_id))

Top product recommendations for Customer 17850
           Avg Quantity
StockCode              
21500         16.666667
84077         16.000000
21181         14.000000
21733         13.333333
20711         12.666667
